In [1]:
print("hi")

hi


In [1]:
import json
import re
import os
import io
import cv2
import numpy as np
import pdfplumber
import torch
from PIL import Image
from pypdf import PdfReader, PdfWriter          # BSD-3-Clause
from reportlab.pdfgen import canvas as pdfcanvas  # open-source ReportLab, BSD-style license
import pypdfium2 as pdfium
# from pipe_fn import pipe
from output_utils import save_split_output 

from transformers import pipeline
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)





DATE_ROW_PATTERN = re.compile(r"^\d{2}/\d{2}/\d{2}$")   # Ameritas uses 2-digit years, e.g. 02/23/26
PATIENT_PATTERN  = re.compile(r"^Patient$")
PROC_CODE_PATTERN = re.compile(r"^D\d{4}$")             # strict "D" + 4 digits
MONEY_TOKEN_PATTERN = re.compile(r"^-?\$?[\d,]+\.\d{2}$")


def count_rows_in_range(words, top, bottom):
    """Number of distinct service-date lines whose y falls within [top, bottom]."""
    rows = set()
    for w in words:
        if top <= w["top"] <= bottom and DATE_ROW_PATTERN.match(w["text"]):
            rows.add(round(w["top"], 1))
    return len(rows)


def find_region_end(page, words):
    """
    Returns a dict with a 'top' key marking where this crop's ROI should end.

    Preferred anchor: the "REMARKS"/"REMARK" legend printed at the bottom of
    the page. NOT every page prints this legend -- pages whose claims don't
    use any remark codes skip it entirely, so we fall back to the bottom of
    the last line of extracted text on the page instead of skipping the
    page outright.
    """
    end = page.search("REMARKS") or page.search("REMARK")
    if end:
        return end[0]

    if not words:
        return {"top": page.height}

    last_word = max(words, key=lambda w: w["bottom"])
    return {"top": min(last_word["bottom"] + 6, page.height)}


def extract_amount_pd(words):
    """
    Reads the single "AMOUNT PD" value directly from the page's digital text
    (never from the VLM) -- it's a plain, unambiguous, page-level number and
    reading it as text is strictly more reliable than an image read.

    Two layouts have been observed:
      1. Value printed on the SAME line, to the right of "PD"
         (e.g. "AMOUNT PD          343.00").
      2. Value printed a couple of lines BELOW the "AMOUNT" / "PD" labels,
         right-aligned under a reprinted PAYABLE-style column.

    We check same-line first (closest match, most common on multi-patient
    pages), then fall back to the NEAREST money-shaped token below the
    label (nearest, not furthest -- picking the furthest risked grabbing
    an unrelated number lower on the page).
    """
    amount_word, pd_word = None, None
    for i, w in enumerate(words):
        if w["text"].upper() == "AMOUNT" and i + 1 < len(words) and words[i + 1]["text"].upper() == "PD":
            amount_word = w
            pd_word = words[i + 1]
            break
    if pd_word is None:
        return ""

    # 1) same line, to the right of "PD"
    same_line = [
        w for w in words
        if abs(w["top"] - pd_word["top"]) <= 2
        and w["x0"] > pd_word["x1"]
        and MONEY_TOKEN_PATTERN.match(w["text"])
    ]
    if same_line:
        best = min(same_line, key=lambda w: w["x0"])
        return best["text"].replace("$", "").replace(",", "")

    # 2) nearest money-shaped token strictly below the label
    below = [
        w for w in words
        if w["top"] > pd_word["top"] + 2 and MONEY_TOKEN_PATTERN.match(w["text"])
    ]
    if below:
        nearest = min(below, key=lambda w: w["top"])
        return nearest["text"].replace("$", "").replace(",", "")

    return ""


# =========================================================================
# CROP + GRID + PER-PATIENT SPLIT
# =========================================================================
def crop_and_grid_pages(pdf_path, output_dir="Ameritas_output", scale=4):
    """
    For every page: find header word positions, compute exact column midpoints
    + row separator y-positions, draw them onto the PDF page, rasterize the
    page, crop the "Your name" -> end-of-content ROI, then SPLIT that ROI into
    one sub-image per patient block (header stitched on top of each) so the
    model is only ever shown one patient at a time.

    Returns a flat list of per-patient crop items, each carrying the shared
    page-level amount_pd (read from text) and that patient's own expected row
    count (ground truth from pdfplumber, for the VLM retry loop to check).
    """
    os.makedirs(output_dir, exist_ok=True)
    patient_crops = []

    with pdfplumber.open(pdf_path) as pdf:
        base_reader = PdfReader(pdf_path)

        for page_num, page in enumerate(pdf.pages, start=1):
            print(f"\n📄 Processing Page {page_num}")

            words = page.extract_words()
            if not words:
                print("   (no text on this page — skipping)")
                continue

            # ---- horizontal (row) separator anchors ----
            horizontal_lines = []
            for w in words:
                txt = w["text"].upper()

                if txt == "PAYABLE":
                    horizontal_lines.append(w["bottom"] + 12)
                elif txt == "PATIENT":
                    horizontal_lines.append(w["top"] - 4)
                elif txt.startswith("MEMBER:"):
                    horizontal_lines.append(w["bottom"] + 6)
                elif txt == "CLAIM":
                    idx = words.index(w)
                    if idx + 1 < len(words) and words[idx + 1]["text"].upper() == "PROCESSED":
                        horizontal_lines.append(w["top"] - 5)
                elif txt == "SUB-TOTAL":
                    horizontal_lines.append(w["top"] - 5)
                elif txt == "EXCLUDED":
                    horizontal_lines.append(w["top"] - 2)
                elif txt == "TOTAL":
                    idx = words.index(w)
                    nxt = words[idx + 1]["text"].upper() if idx + 1 < len(words) else ""
                    if nxt == "PAID":
                        horizontal_lines.append(w["top"] - 5)
                    else:
                        horizontal_lines.append(w["bottom"] + 3)
                elif "====" in txt:
                    horizontal_lines.append(w["top"])
                elif DATE_ROW_PATTERN.match(w["text"]):
                    horizontal_lines.append(w["bottom"] + 3)

            # ---- locate the table header row ----
            th = next((w for w in words if w["text"].upper() == "TH"), None)
            if th is None:
                print(f"   Page {page_num}: table header 'TH' not found — skipping")
                continue

            header_top = th["top"]
            header_words = sorted(
                [w for w in words if header_top - 3 <= w["top"] <= header_top + 18],
                key=lambda x: x["x0"],
            )

            def first(text):
                for w in header_words:
                    if w["text"].upper() == text:
                        return w
                return None

            def nth(text, n):
                cnt = 0
                for w in header_words:
                    if w["text"].upper() == text:
                        cnt += 1
                        if cnt == n:
                            return w
                return None

            header = {
                "TH": first("TH"), "PROC": first("PROC"), "PAY": first("PAY"),
                "DATE": first("DATE"), "SERVICE1": nth("SERVICE", 1),
                "DESCRIPTION": first("DESCRIPTION"),
                "BENEFIT": first("BENEFIT"), "TYPE": first("TYPE"),
                "SUBMITD": first("SUBMITD"), "CHARGES1": nth("CHARGES", 1),
                "ELIG": first("ELIG"), "CHARGES2": nth("CHARGES", 2), "CHARGES3": nth("CHARGES", 3),
                "COVERED": first("COVERED"), "AMOUNT": first("AMOUNT"),
                "RMRK": first("RMRK"), "CODE3": nth("CODE", 3),
                "MINUS": first("MINUS"), "DEDCT": first("DEDCT"),
                "REMAIN": first("REMAIN"), "PERCENT": first("%"),
                "PAYABLE": first("PAYABLE"),
            }

            missing = [k for k, v in header.items() if v is None and k != "SERVICE1"]
            if missing:
                print(f"   ⚠️  Page {page_num}: missing header word(s) {missing} — skipping")
                continue

            # ---- region anchors ----
            start = page.search("Your name")
            if not start:
                print("   ⚠️  'Your name' not found — skipping")
                continue
            start_crop = start[0]
            end_crop = find_region_end(page, words)
            region_top = start_crop["top"]
            region_bottom = end_crop["top"]

            # ---- vertical (column) separators, from exact header word gaps ----
            columns = [
                (header["PROC"]["x1"], header["PAY"]["x0"]),
                (header["PAY"]["x1"], header["DATE"]["x0"]),
                (header["DESCRIPTION"]["x1"], header["BENEFIT"]["x0"]),
                (header["TYPE"]["x1"] + 5, header["SUBMITD"]["x0"]),
                (header["CHARGES1"]["x1"], header["ELIG"]["x0"]),
                (header["CHARGES2"]["x1"], header["COVERED"]["x0"]),
                (header["AMOUNT"]["x1"], header["RMRK"]["x0"]),
                (header["CODE3"]["x1"], header["MINUS"]["x0"]),
                (header["DEDCT"]["x1"], header["REMAIN"]["x0"]),
                (header["CHARGES3"]["x1"], header["PERCENT"]["x0"]),
                (header["PERCENT"]["x1"], header["PAYABLE"]["x0"]),
            ]

            x_lines = [header["TH"]["x0"] - 6]
            for right_edge, next_left in columns:
                x_lines.append((right_edge + next_left) / 2)
            x_lines.append(header["PAYABLE"]["x1"] + 6)

            table_top = header["TH"]["top"] - 4
            table_bottom = region_bottom - 3

            page_width = float(page.width)
            page_height = float(page.height)
            left_x = header["TH"]["x0"] - 6
            right_x = header["PAYABLE"]["x1"] + 6

            overlay_buffer = io.BytesIO()
            oc = pdfcanvas.Canvas(overlay_buffer, pagesize=(page_width, page_height))
            oc.setLineWidth(0.7)
            oc.setStrokeColorRGB(0, 0, 0)

            fill_missing_benefit_type(words, header, oc, page_height)

            submitd_elig_x = (header["CHARGES1"]["x1"] + header["ELIG"]["x0"]) / 2
            for x in x_lines:
                if abs(x - submitd_elig_x) < 1:
                    y = table_top
                    while y < table_bottom:
                        oc.circle(x, page_height - y, 0.4, stroke=1, fill=1)
                        y += 3
                else:
                    oc.line(x, page_height - table_top, x, page_height - table_bottom)

            for y in horizontal_lines:
                oc.line(left_x, page_height - y, right_x, page_height - y)

            oc.save()
            overlay_buffer.seek(0)
            overlay_page = PdfReader(overlay_buffer).pages[0]

            writer = PdfWriter()
            writer.add_page(base_reader.pages[page_num - 1])
            writer.pages[0].merge_page(overlay_page)

            merged_buffer = io.BytesIO()
            writer.write(merged_buffer)
            merged_buffer.seek(0)

            pdfium_doc = pdfium.PdfDocument(merged_buffer.read())
            pdfium_page = pdfium_doc[0]
            bitmap = pdfium_page.render(scale=scale)
            pil_image = bitmap.to_pil().convert("RGB")
            pdfium_doc.close()

            img = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

            crop_top_px = int(region_top * scale)
            crop_bottom_px = int(region_bottom * scale)
            full_crop = img[crop_top_px:crop_bottom_px, :]

            # ---- page-level amount_pd, read directly from text ----
            amount_pd_value = extract_amount_pd(words)

            # ---- find each patient block's boundaries ----
            patient_tops = sorted(set(round(w["top"], 1) for w in words
                                       if PATIENT_PATTERN.match(w["text"]) and region_top <= w["top"] <= region_bottom))
            divider_tops = sorted(w["top"] for w in words
                                   if "====" in w["text"] and region_top <= w["top"] <= region_bottom)

            if not patient_tops:
                print(f"   ⚠️  Page {page_num}: no 'Patient (...)' rows found — skipping")
                continue

            # header snapshot: from the top of the crop down to just above the first patient row
            header_bottom_page_y = patient_tops[0] - 4
            header_bottom_px = max(int((header_bottom_page_y - region_top) * scale), 1)
            header_img = full_crop[0:header_bottom_px, :]

            blocks = []
            for p_top in patient_tops:
                later_dividers = [d for d in divider_tops if d > p_top]
                block_end = min(later_dividers) if later_dividers else region_bottom
                blocks.append((p_top, block_end))

            for i, (b_start, b_end) in enumerate(blocks, start=1):
                px_start = max(int((max(b_start - 4, region_top) - region_top) * scale), 0)
                px_end = min(int((min(b_end + 4, region_bottom) - region_top) * scale), full_crop.shape[0])
                patient_body_img = full_crop[px_start:px_end, :]

                sub_img = np.vstack([header_img, patient_body_img])

                image_path = os.path.join(output_dir, f"page_{page_num}_patient_{i}.png")
                cv2.imwrite(image_path, sub_img)
                print(f"✅ Saved: {image_path}")

                expected_rows = count_rows_in_range(words, b_start, b_end)

                patient_crops.append({
                    "page": page_num,
                    "patient_index": i,
                    "image_path": image_path,
                    "expected_rows": expected_rows,
                    "amount_pd": amount_pd_value,
                    "page_total_patients": len(blocks),
                })

    return patient_crops


def fill_missing_benefit_type(words, header, oc, page_height):
    benefit_left = header["BENEFIT"]["x0"]
    benefit_right = header["TYPE"]["x1"] + 5

    oc.setFont("Helvetica", 8)

    service_rows = [w for w in words if re.fullmatch(r"D\d{4}", w["text"])]

    for proc in service_rows:
        row_top = proc["top"] - 2
        row_bottom = proc["bottom"] + 2

        benefit_words = [
            w for w in words
            if (benefit_left <= w["x0"] <= benefit_right and row_top <= w["top"] <= row_bottom)
        ]

        if not benefit_words:
            x = (benefit_left + benefit_right) / 2
            y = (row_top + row_bottom) / 2
            oc.drawCentredString(x, page_height - y - 2, "BASIC")
            print(f"Inserted BASIC for {proc['text']}")


# =========================
# PROMPT (single patient per call)
# =========================
def build_prompt(pdf_name: str) -> str:
    return f"""
Extract structured data for the ONE patient shown in this image. This image contains the shared
column header at the top, followed by exactly ONE patient's block: a "Patient (SELF):" or
"Patient (DEPENDENT):" line with a claim number, that patient's service rows, that patient's
"TOTAL <BENEFIT TYPE>" row(s), and that patient's SUB-TOTAL / BILLABLE AMOUNT TO PATIENT / TOTAL
PAID lines. There is no other patient in this image -- do not invent one, and do not repeat any
row or line twice.

===========================================================================
RULE SET
===========================================================================

1. PATIENT HEADER FIELDS
   - patient_name : text after "Patient (SELF):" or "Patient (DEPENDENT):"
   - relationship : "SELF" or "DEPENDENT"
   - claim_number : the value after "Claim #:" on that same line (e.g. "04-22-675-9200-0").

2. SERVICE ROWS BELONG TO A "TOTAL <BENEFIT TYPE>" GROUP
   Services may span multiple benefit types (e.g. some rows "PREVENT", others "MAJOR", others
   "BASIC"). Each group of same-type service rows is immediately followed by ONE
   "TOTAL <BENEFIT TYPE>" row. Do not merge, skip, or reorder rows. Do not treat a "TOTAL" row
   as a service row.

3. FIELDS THAT ONLY EVER APPEAR ON A "TOTAL <BENEFIT TYPE>" ROW — NEVER ON A SERVICE ROW:
   - minus_dedct (MINUS DEDCT)
   - remain_charges (REMAIN CHARGES)
   - percent (%)
   - payable (PAYABLE)
   These four columns are printed BLANK on every individual service row and only filled in on
   the "TOTAL <TYPE>" row beneath the group. If you do not see a printed value in that exact
   cell of that exact service row, the field is "". NEVER copy a value down from the TOTAL row
   into a service row above it.

4. PATIENT-LEVEL FIELDS (extract literally)
   - sub_total: the number on the "SUB-TOTAL" line.
   - excluded_amount: if (and only if) a line reading "EXCLUDED AMOUNT" appears between
     SUB-TOTAL and TOTAL PAID, extract its value (printed as a negative number, e.g.
     "-3127.50"). If it is not printed, return "".
   - billable_amount_to_patient: the number printed directly after "BILLABLE AMOUNT TO
     PATIENT". Do NOT take this value from PAYABLE, SUB-TOTAL, TOTAL PAID, or EXCLUDED AMOUNT
     — it is its own distinct printed number, and it may legitimately be $0.00.
   - total_paid: the number printed on "TOTAL PAID".

5. MONEY / PERCENT FORMATTING
   - Money: strip "$" and "," but keep decimals and any leading "-" sign, return as a string,
     e.g. "$1,234.00" -> "1234.00", "-$3,127.50" -> "-3127.50".
   - Percent: keep the "%" sign, e.g. "100%", "80%", "0%".
   - If a field is blank in the image, return "".
   - Never infer or fabricate a missing value.

6. BENEFIT TYPE — STRICT, PER ROW ONLY
   Read ONLY the BENEFIT TYPE cell of THIS service row. Do not infer it from the procedure
   code, the service description, a neighboring row, or a TOTAL row.
   - "BASIC" or blank cell -> "BASIC"
   - "PREVENT" -> "PREVENT"
   - "MAJOR" -> "MAJOR"

7. PROC CODE — STRICT FORMAT VALIDATION
   A valid proc code is exactly the letter "D" followed by 4 digits (e.g. "D0120", "D2740").
   No other format is valid. Never take a value from the service-description column as a proc
   code. If a row's PROC CODE cell does not match this exact pattern, DO NOT include that row
   in "services" at all. The same proc code CAN legitimately repeat several times for one
   patient (e.g. six separate "D2740" crown rows on six different teeth) — each physically
   printed line is its own real service row.

8. SERVICE ROW COUNT — HIGHEST PRIORITY
   A service row exists ONLY if there is exactly one visible, physically printed PROC CODE
   (D + 4 digits) in the PROC CODE column. Never duplicate, repeat, or fabricate a row. Stop
   as soon as the visible PROC CODE rows are exhausted.

9. DATE OF SERVICE
   Extract ONLY the date token itself (e.g. "06/26/25") from the DATE OF SERVICE cell.

===========================================================================
COLUMN MAPPING
===========================================================================

Service rows:
  proc_code          <- PROC CODE
  service_date       <- DATE OF SERVICE (date token only)
  benefit_type       <- BENEFIT TYPE
  submitted_charges  <- SUBMITD CHARGES
  elig_charges       <- ELIG CHARGES
  covered_amount     <- COVERED AMOUNT
  rmrk_code          <- RMRK CODE
  minus_dedct        <- always "" on a service row (see rule 3)
  remain_charges     <- always "" on a service row (see rule 3)
  payable            <- always "" on a service row (see rule 3)

TOTAL <BENEFIT TYPE> row (-> one benefit_totals object per distinct benefit type used):
  benefit_type       <- the type name on the "TOTAL <TYPE>" row
  submitted_charges  <- SUBMITD CHARGES on that TOTAL row
  elig_charges       <- ELIG CHARGES on that TOTAL row
  covered_amount     <- COVERED AMOUNT on that TOTAL row
  minus_dedct        <- MINUS DEDCT on that TOTAL row
  remain_charges     <- REMAIN CHARGES on that TOTAL row
  percent            <- % on that TOTAL row (keep the "%" sign)
  payable            <- PAYABLE on that TOTAL row
===========================================================================
OUTPUT JSON SCHEMA (valid JSON only — no trailing commas, no markdown, no prose)
===========================================================================

{{

  "patient_name": {{
    "value": "",
    "confidence": 0.0
  }},

  "relationship": {{
    "value": "",
    "confidence": 0.0
  }},

  "claim_number": {{
    "value": "",
    "confidence": 0.0
  }},

  "services": [
    {{

      "proc_code": {{
        "value": "",
        "confidence": 0.0
      }},

      "service_date": {{
        "value": "",
        "confidence": 0.0
      }},

      "benefit_type": {{
        "value": "",
        "confidence": 0.0
      }},

      "submitted_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "elig_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "covered_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "rmrk_code": {{
        "value": "",
        "confidence": 0.0
      }},

      "minus_dedct": {{
        "value": "",
        "confidence": 0.0
      }},

      "remain_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "payable": {{
        "value": "",
        "confidence": 0.0
      }}

    }}
  ],

  "benefit_totals": [
    {{

      "benefit_type": {{
        "value": "",
        "confidence": 0.0
      }},

      "submitted_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "elig_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "covered_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "minus_dedct": {{
        "value": "",
        "confidence": 0.0
      }},

      "remain_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "percent": {{
        "value": "",
        "confidence": 0.0
      }},

      "payable": {{
        "value": "",
        "confidence": 0.0
      }}

    }}
  ],

  "sub_total": {{
    "value": "",
    "confidence": 0.0
  }},

  "excluded_amount": {{
    "value": "",
    "confidence": 0.0
  }},

  "billable_amount_to_patient": {{
    "value": "",
    "confidence": 0.0
  }},

  "total_paid": {{
    "value": "",
    "confidence": 0.0
  }}

}}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

===========================================================================
VALIDATION RULES (self-check before answering)
===========================================================================
- Number of service objects must equal the number of visible rows whose PROC CODE column
  contains a valid "D" + 4-digit code.
- Rows without a valid proc code must be excluded from "services" entirely.
- Every service row's minus_dedct, remain_charges, and payable must be "".
- Do not include "TOTAL" / "SUB-TOTAL" / "EXCLUDED AMOUNT" rows inside "services".
- "benefit_type" in service rows must be exactly "PREVENT", "MAJOR", or "BASIC".
- Output ONLY JSON for this ONE patient. Do not wrap it in a list, and do not add any other
  patient.
"""


# =========================
# SANITIZATION (hard guarantees, independent of what the model actually returns)
# =========================
def sanitize_patient(patient: dict) -> dict:
    """
    Enforces, in code, the structural facts the model doesn't reliably follow on its own:
      1. minus_dedct / remain_charges / payable are forced blank on every service row.
      2. A service row is only kept if its proc_code matches D+4digits.
    """
    clean_services = []
    for svc in patient.get("services", []):
        proc = str(svc.get("proc_code", "")).strip()
        if not PROC_CODE_PATTERN.match(proc):
            continue
        svc["minus_dedct"] = ""
        svc["remain_charges"] = ""
        svc["payable"] = ""
        clean_services.append(svc)
    patient["services"] = clean_services
    return patient


# =========================
# AMOUNT / VALIDATION HELPERS
# =========================
def parse_amount(x) -> float:
    if x is None or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace("$", "").replace(",", "").replace("%", "").strip())
    except ValueError:
        return 0.0


SERVICE_SUM_FIELDS = ["submitted_charges", "elig_charges", "covered_amount"]


def services_are_empty(patient: dict) -> bool:
    for svc in patient.get("services", []):
        for f in ("submitted_charges", "elig_charges", "covered_amount"):
            if str(svc.get(f, "")).strip() not in ("", "0.00", "0"):
                return False
    return True


def validate_patient(patient: dict) -> tuple:
    """
    Validates ONE patient block:

      A) Per benefit-type sums: sum this patient's own service rows of each benefit type and
         compare submitted_charges / elig_charges / covered_amount against the matching TOTAL
         row.

      B) Per benefit-type TOTAL-row remain_charges formula (using only numbers already on the
         TOTAL row itself):
           remain_charges == covered_amount - minus_dedct

         NOTE: "payable" is extracted and reported as-is, but is NOT cross-checked against
         remain_charges * percent / 100. Real EOPs can apply annual-maximum caps, COB
         reductions, or other plan-specific adjustments to payable that aren't printed
         anywhere else on the page -- there's nothing to compute it from, so this formula
         check produced false-positive errors on otherwise-correct extractions.

      C) Patient-level consistency:
           total_paid == sub_total + excluded_amount   (excluded_amount defaults to 0)
           sub_total  == sum(payable across benefit_totals)
           billable_amount_to_patient == sum(elig_charges) - total_paid
    """
    services = patient.get("services", [])
    benefit_totals = patient.get("benefit_totals", [])
    name = patient.get("patient_name", "UNKNOWN")
    claim_number = patient.get("claim_number", "")
    errors = []
    log = f"\n🔍 Validation for [{name}] (claim {claim_number or 'n/a'})\n" + "-" * 80

    if not services:
        return False, log + "\n❌ No services found", [{"error": "empty_services"}]

    grouped = {}
    for s in services:
        bt = s.get("benefit_type", "").strip()
        grouped.setdefault(bt, []).append(s)

    for bt_row in benefit_totals:
        bt = bt_row.get("benefit_type", "").strip()
        rows = grouped.get(bt, [])

        computed = {
            "submitted_charges": round(sum(parse_amount(r.get("submitted_charges", "")) for r in rows), 2),
            "elig_charges": round(sum(parse_amount(r.get("elig_charges", "")) for r in rows), 2),
            "covered_amount": round(sum(parse_amount(r.get("covered_amount", "")) for r in rows), 2),
        }
        for f in SERVICE_SUM_FIELDS:
            extracted = round(parse_amount(bt_row.get(f, "")), 2)
            match = abs(computed[f] - extracted) <= 0.01
            icon = "✅" if match else "❌"
            log += f"\n{icon} [{bt}] {f:20s} computed={computed[f]:<10} | extracted={extracted:<10}"
            if not match:
                errors.append({
                    "type": "benefit_total_mismatch", "benefit_type": bt, "field": f,
                    "computed": computed[f], "extracted": extracted,
                })

        extracted_covered = round(parse_amount(bt_row.get("covered_amount", "")), 2)
        extracted_minus = round(parse_amount(bt_row.get("minus_dedct", "")), 2)
        extracted_remain = round(parse_amount(bt_row.get("remain_charges", "")), 2)
        expected_remain = round(extracted_covered - extracted_minus, 2)
        remain_match = abs(expected_remain - extracted_remain) <= 0.01
        log += (f"\n{'✅' if remain_match else '❌'} [{bt}] remain_charges(=covered-minus_dedct) "
                f"computed={expected_remain:<10} | extracted={extracted_remain:<10}")
        if not remain_match:
            errors.append({
                "type": "remain_charges_formula_mismatch", "benefit_type": bt,
                "computed": expected_remain, "extracted": extracted_remain,
            })

        # "payable" is extracted-only -- no formula cross-check (see validate_patient docstring).
        extracted_payable = round(parse_amount(bt_row.get("payable", "")), 2)
        log += f"\nℹ️  [{bt}] payable (extracted) = {extracted_payable}"

    extracted_sub_total = round(parse_amount(patient.get("sub_total", "")), 2)
    extracted_total_paid = round(parse_amount(patient.get("total_paid", "")), 2)
    extracted_excluded = round(parse_amount(patient.get("excluded_amount", "")), 2)

    expected_total_paid = round(extracted_sub_total + extracted_excluded, 2)
    paid_match = abs(expected_total_paid - extracted_total_paid) <= 0.01
    log += (f"\n{'✅' if paid_match else '❌'} total_paid(=sub_total+excluded_amount) "
            f"computed={expected_total_paid:<10} | extracted={extracted_total_paid:<10}")
    if not paid_match:
        errors.append({
            "type": "total_paid_formula_mismatch",
            "computed": expected_total_paid, "extracted": extracted_total_paid,
        })

    payable_sum = round(sum(parse_amount(r.get("payable", "")) for r in benefit_totals), 2)
    sub_vs_payable_match = abs(extracted_sub_total - payable_sum) <= 0.01
    log += (f"\n{'✅' if sub_vs_payable_match else '❌'} sub_total vs sum(payable) "
            f"sub_total={extracted_sub_total:<10} | sum(payable)={payable_sum:<10}")
    if not sub_vs_payable_match:
        errors.append({
            "type": "sub_total_vs_payable_sum_mismatch",
            "sub_total": extracted_sub_total, "sum_payable": payable_sum,
        })

    elig_sum = round(sum(parse_amount(r.get("elig_charges", "")) for r in benefit_totals), 2)
    expected_billable = round(elig_sum - extracted_total_paid, 2)
    billable_extracted = round(parse_amount(patient.get("billable_amount_to_patient", "")), 2)
    billable_match = abs(expected_billable - billable_extracted) <= 0.01
    log += (f"\n{'✅' if billable_match else '❌'} billable_amount_to_patient(=elig_sum-total_paid) "
            f"computed={expected_billable:<10} | extracted={billable_extracted:<10}")
    if not billable_match:
        errors.append({
            "type": "field_mismatch", "field": "billable_amount_to_patient",
            "computed": expected_billable, "extracted": billable_extracted,
        })

    log += "\n" + "-" * 80
    return (len(errors) == 0), log, errors


def validate_page(patients: list, amount_pd: str, expected_patient_count: int) -> tuple:
    """
    Page-level rollup, run after every patient sub-image on a page has been extracted:
      - patient count vs pdfplumber ground truth
      - each patient individually (validate_patient)
      - amount_pd is reported AS EXTRACTED ONLY -- no computed cross-check against
        sum(total_paid) or sum(sub_total+excluded_amount). AMOUNT PD is a page-level
        printed figure; the only thing that is ever "computed" for it is the fallback
        derivation used when it wasn't printed as text at all (see below). When it WAS
        found on the page, we simply pass through what was extracted -- there is no
        validation/error generation against it.

    If amount_pd was NOT found as text (extract_amount_pd returned ""), it is DERIVED as the
    sum of total_paid across this page's patient blocks -- that is literally what AMOUNT PD
    represents when a page has multiple patient blocks, so this is a safe fallback rather than
    a guess. That derived value is not itself "validated" against anything else either --
    it's just used because nothing else was printed to report.

    Returns: (all_valid, full_log, all_errors, resolved_amount_pd, amount_pd_was_derived)
    """
    all_errors = []
    full_log = ""
    all_valid = True

    if len(patients) != expected_patient_count:
        all_errors.append({
            "type": "patient_count_mismatch",
            "expected": expected_patient_count, "extracted": len(patients),
        })
        all_valid = False

    for patient in patients:
        ok, log, errs = validate_patient(patient)
        full_log += log
        all_errors.extend(errs)
        all_valid = all_valid and ok

    amount_pd_found = str(amount_pd).strip() != ""

    if not amount_pd_found:
        # Fallback only: nothing was printed for amount_pd, so derive it as the sum of
        # each patient's TOTAL PAID. This is the ONLY place amount_pd is computed --
        # there is no cross-check/error generated against it either way.
        total_paid_sum = round(sum(parse_amount(p.get("total_paid", "")) for p in patients), 2)
        resolved_amount_pd = f"{total_paid_sum:.2f}"
        amount_pd_derived = True
        full_log += (f"\nℹ️  amount_pd not found on page — derived as sum(total_paid) = "
                     f"{resolved_amount_pd}")
    else:
        # Extracted as-is, no computation, no cross-check, no error.
        resolved_amount_pd = str(amount_pd)
        amount_pd_derived = False
        full_log += f"\nℹ️  amount_pd (extracted) = {resolved_amount_pd}"

    print(full_log)
    return all_valid, full_log, all_errors, resolved_amount_pd, amount_pd_derived


# =========================
# JSON CLEANER
# =========================
def extract_json(text: str) -> dict:
    start = text.find("{")
    end = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in model output")
    return json.loads(text[start:end])


def save_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# =========================
# RETRY HELPER (single patient per call)
# =========================
MAX_RETRIES = 2

def run_model_with_retry(image: Image.Image, prompt: str, expected_rows: int):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    last_parsed = None
    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  🔄 Model attempt {attempt}/{MAX_RETRIES}")
        with torch.no_grad():
            output = pipe(messages, max_new_tokens=12000, temperature=0.0, do_sample=False)

        raw = output[0]["generated_text"]
        if isinstance(raw, list):
            raw = raw[-1]["content"]

        print(raw)

        try:
            parsed = extract_json(raw)
        except Exception as e:
            print(f"  ❌ JSON parse failed on attempt {attempt}: {e}")
            continue

        model_confidence = calculate_model_confidence(parsed)
        parsed = _unwrap_vlm_output(parsed)
        parsed["_model_confidence"] = model_confidence

        parsed = sanitize_patient(parsed)

        total_rows = len(parsed.get("services", []))
        row_ok = (total_rows == expected_rows)
        empty_ok = not services_are_empty(parsed)

        if row_ok and empty_ok:
            print(f"  ✅ Accepted on attempt {attempt}")
            return parsed

        print(f"  ⚠️  attempt {attempt}: rows={total_rows} (expected {expected_rows}), "
              f"all_empty={not empty_ok}")
        last_parsed = parsed

    print(f"  ⚠️  All {MAX_RETRIES} attempts exhausted — using last result")
    return last_parsed


# =========================
# MAIN PIPELINE
# =========================
def run_pipeline(pdf_path: str, output_dir: str = "Ameritas_result_2000", scale: int = 4, company_name = "Ameritas"):
    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]

    base_dir = os.path.join(output_dir, pdf_name)
    pdf_full_name = os.path.basename(pdf_path)  
    cropped_dir = os.path.join(base_dir, "cropped_images")
    json_output_path = os.path.join(base_dir, f"{pdf_name}_output.json")

    from output_utils import SUCCESS_DIR, FAILED_DIR   # ADD
    already_success = os.path.exists(os.path.join(SUCCESS_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    already_failed  = os.path.exists(os.path.join(FAILED_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    if already_success or already_failed:
        print(f"⏭️  Skipping {pdf_name} — output already exists")
        return None


    os.makedirs(base_dir, exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    # Crop + grid-draw + split into one image per patient block
    patient_crop_items = crop_and_grid_pages(pdf_path, output_dir=cropped_dir, scale=scale)

    final_prompt = build_prompt(pdf_name)

    # Run the model once per patient sub-image, then group results back up by page
    pages = {}  # page_num -> {"amount_pd": ..., "expected_patient_count": ..., "patients": [...]}

    for idx, item in enumerate(patient_crop_items):
        page_num = item["page"]
        img_path = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"\nProcessing page {page_num}, patient {item['patient_index']}/{item['page_total_patients']} "
              f"(expecting {expected_rows} rows)")

        image_pil = Image.open(img_path).convert("RGB")
        parsed_patient = run_model_with_retry(image_pil, final_prompt, expected_rows)

        pages.setdefault(page_num, {
            "amount_pd": item["amount_pd"],
            "expected_patient_count": item["page_total_patients"],
            "patients": [],
        })

        if parsed_patient is None:
            print(f"❌ Skipping patient {item['patient_index']} on page {page_num} — model returned nothing usable")
            continue

        pages[page_num]["patients"].append(parsed_patient)

    all_crops_output = []
    for page_num, page_data in sorted(pages.items()):
        patients = page_data["patients"]
        amount_pd = page_data["amount_pd"]
        expected_patient_count = page_data["expected_patient_count"]

        is_valid, log, errors, resolved_amount_pd, amount_pd_derived = validate_page(
            patients, amount_pd, expected_patient_count
        )

        all_crops_output.append({
            "patients": patients,
            "amount_pd": resolved_amount_pd,
            "amount_pd_derived": amount_pd_derived,
            "_page": page_num,
            "_table": 1,
            "validation": {"status": is_valid, "errors": errors},
        })

    all_patients_flat = [p for page_data in pages.values() for p in page_data["patients"]]
    confidence_score = calculate_eob_confidence(all_patients_flat)

    for page_table in all_crops_output:
        for p in page_table["patients"]:
            p.pop("_model_confidence", None)

        

    final = [
        {
            "eob_id": pdf_name,
            "file_name": pdf_full_name,          # ADD
            "confidence_score": confidence_score,
            **table
        }
        for table in all_crops_output
    ]

    success_path, failed_path = save_split_output(
        final,
        company_name=company_name,
        pdf_name=pdf_name,
        pdf_path=pdf_path,
        cropped_dir=cropped_dir,
    )

    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 17:44:43.919000 3338982 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 17:44:43.933000 3338982 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

## Test 1

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/AMERITAS/pdfs"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_837759737.pdf

📄 Processing Page 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Saved: Ameritas_result_2000/837759737/cropped_images/page_1_patient_1.png

Processing page 1, patient 1/1 (expecting 2 rows)
  🔄 Model attempt 1/2


[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "patient_name": {
    "value": "MARCIA REEVES",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-601-9335-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "02/19/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",
        "confidence":

...

```json
{
  "patient_name": {
    "value": "KATINA J IZURIETA",
    "confidence": 1.0
  },
  "relationship": {
    "value": "DEPENDENT",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-605-1236-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "02/24/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",
        "con

[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "patient_name": {
    "value": "LISA A EASTON",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-610-0078-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "03/02/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",
        "confidence":

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  🔄 Model attempt 1/2
```json
{
  "patient_name": {
    "value": "DEEDRA BRYANT",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-615-3349-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "03/09/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "95.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",

# Test 2

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/AMERITAS/pdfs"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_837759737.pdf

📄 Processing Page 1
✅ Saved: Ameritas_result_2000/837759737/cropped_images/page_1_patient_1.png

Processing page 1, patient 1/1 (expecting 2 rows)


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  🔄 Model attempt 1/2
```json
{
  "patient_name": {
    "value": "MARCIA REEVES",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-601-9335-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "02/19/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Saved: Ameritas_result_2000/845701432/cropped_images/page_1_patient_1.png
✅ Saved: Ameritas_result_2000/845701432/cropped_images/page_1_patient_2.png

📄 Processing Page 2
   Page 2: table header 'TH' not found — skipping

Processing page 1, patient 1/2 (expecting 4 rows)
  🔄 Model attempt 1/2


...

```json
{
  "patient_name": {
    "value": "MELANIE EARLY",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-603-9124-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "02/20/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",
        "confidence":

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  🔄 Model attempt 1/2
```json
{
  "patient_name": {
    "value": "LISA A EASTON",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-610-0078-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "03/02/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 0.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 0.0
      },
      "remain_charges": {
        "value": "",

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  🔄 Model attempt 1/2
```json
{
  "patient_name": {
    "value": "DEEDRA BRYANT",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-615-3349-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "03/09/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "95.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/AMERITAS/pdfs/Pmt_EOP_839932487.pdf")


📄 Processing Page 1
✅ Saved: Ameritas_result_2000/839932487/cropped_images/page_1_patient_1.png

📄 Processing Page 2


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.


✅ Saved: Ameritas_result_2000/839932487/cropped_images/page_2_patient_1.png

📄 Processing Page 3
✅ Saved: Ameritas_result_2000/839932487/cropped_images/page_3_patient_1.png
✅ Saved: Ameritas_result_2000/839932487/cropped_images/page_3_patient_2.png

📄 Processing Page 4
   Page 4: table header 'TH' not found — skipping

Processing page 1, patient 1/1 (expecting 2 rows)
  🔄 Model attempt 1/2


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "patient_name": {
    "value": "KEELY R GREENE",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-603-9611-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "02/23/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",
        "confidence"

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "patient_name": {
    "value": "JOSEPH R PHELAN",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-603-9138-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "02/20/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",
        "confidence

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=12000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "patient_name": {
    "value": "MELANIE EARLY",
    "confidence": 1.0
  },
  "relationship": {
    "value": "SELF",
    "confidence": 1.0
  },
  "claim_number": {
    "value": "04-22-603-9124-0",
    "confidence": 1.0
  },
  "services": [
    {
      "proc_code": {
        "value": "D0120",
        "confidence": 1.0
      },
      "service_date": {
        "value": "02/20/26",
        "confidence": 1.0
      },
      "benefit_type": {
        "value": "PREVENT",
        "confidence": 1.0
      },
      "submitted_charges": {
        "value": "89.00",
        "confidence": 1.0
      },
      "elig_charges": {
        "value": "38.00",
        "confidence": 1.0
      },
      "covered_amount": {
        "value": "38.00",
        "confidence": 1.0
      },
      "rmrk_code": {
        "value": "",
        "confidence": 1.0
      },
      "minus_dedct": {
        "value": "",
        "confidence": 1.0
      },
      "remain_charges": {
        "value": "",
        "confidence":

[{'eob_id': '839932487',
  'file_name': 'Pmt_EOP_839932487.pdf',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'KEELY R GREENE',
    'relationship': 'SELF',
    'claim_number': '04-22-603-9611-0',
    'services': [{'proc_code': 'D0120',
      'service_date': '02/23/26',
      'benefit_type': 'PREVENT',
      'submitted_charges': '89.00',
      'elig_charges': '38.00',
      'covered_amount': '38.00',
      'rmrk_code': '',
      'minus_dedct': '',
      'remain_charges': '',
      'payable': ''},
     {'proc_code': 'D1110',
      'service_date': '02/23/26',
      'benefit_type': 'PREVENT',
      'submitted_charges': '149.00',
      'elig_charges': '85.00',
      'covered_amount': '85.00',
      'rmrk_code': '',
      'minus_dedct': '',
      'remain_charges': '',
      'payable': ''}],
    'benefit_totals': [{'benefit_type': 'PREVENT',
      'submitted_charges': '238.00',
      'elig_charges': '123.00',
      'covered_amount': '123.00',
      'minus_dedct': '0.00',
     